# Notebook 03 — Partial Correlations: Risk Factors × PC1_pace / PC1_status

**Purpose:** Compute partial correlations between ~65 environmental and biological risk factors and
each of the two brain scores (`PC1_pace`, `PC1_status`), with appropriate covariates and FDR correction.

**Inputs:**
- `data/covariates/env_risk_factors_final.csv` — environmental + biological risk factors (has `participant_id` column)
- `data/outcomes/mh_all.csv` — mental health at baseline (00A) and follow-up (04A); ID-indexed
- `data/pca/finalscore.csv` — PC1_pace, PC1_status; ID-indexed

**Outputs (saved to `outputs/partial_correlations/`):**
- `partial_corr_pace.csv`
- `partial_corr_status.csv`

## Configuration

In [ ]:
import os

# ── Paths (all relative — no personal paths) ──────────────────────────────────
ENV_CSV     = 'data/covariates/env_risk_factors_final.csv'  # environmental + biological risk factors
MH_CSV      = 'data/outcomes/mh_all.csv'                    # mental health at 00A and 04A; ID index
PCA_CSV     = 'data/pca/finalscore.csv'                     # PC1_pace, PC1_status; ID index
OUTPUT_DIR  = 'outputs/partial_correlations'

WAVE_BASE   = '00A'
WAVE_TARGET = '04A'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory ready: {OUTPUT_DIR}')

## Imports

In [ ]:
import re
import ast
import numpy as np
import pandas as pd
import pingouin as pg
from statsmodels.stats.multitest import multipletests

## Step 1 — Define label mappings

In [ ]:
LABEL_MAP = {
    'ab_g_stc__design_id__fam': 'Family ID',
    'ab_g_stc__cohort_ethn': 'Ethnicity',
    'ab_g_stc__cohort_race__nih': 'Race',
    'ab_g_stc__cohort_sex': 'Sex',
    'ab_g_dyn__visit_age': 'Age (Baseline)',
    'ph_y_anthr__waist_001': 'Waist',
    'BMI': 'BMI',
    'pds_total': 'Pubertal Stage',
    'ph_p_dhx_012': 'Breastfeeding Duration',
    'ph_p_dhx_013': 'Motor Development',
    'ph_p_dhx_014': 'Speech Development',
    'ph_p_dhx_003__01': 'Maternal Age',
    'ph_p_dhx_004__01': 'Paternal Age',
    'ph_p_dhx__birth_001': 'Prematurity',
    'ph_p_dhx__birth_003': 'Cyanosis at Birth',
    'ph_p_dhx__birth_004': 'Bradycardia at Birth',
    'ph_p_dhx__birth_005': 'Apnea at Birth',
    'ph_p_dhx__birth_006': 'Neonatal Convulsions',
    'ph_p_dhx__birth_007': 'Jaundice Treatment',
    'ph_p_dhx__birth_008': 'Oxygen Required',
    'ph_p_dhx__birth_009': 'Blood Transfusion',
    'ph_p_dhx__birth_010': 'Rh Incompatibility',
    'ph_p_dhx_006': 'Planned Pregnancy',
    'ph_p_dhx__med_001': 'Nausea/Vomiting',
    'ph_p_dhx__med_002': 'Heavy Bleeding',
    'ph_p_dhx__med_003': 'Pre-eclampsia',
    'ph_p_dhx__med_004': 'Gallbladder Attack',
    'ph_p_dhx__med_005': 'Proteinuria',
    'ph_p_dhx__med_006': 'Rubella',
    'ph_p_dhx__med_007': 'Anemia',
    'ph_p_dhx__med_008': 'Infection',
    'ph_p_dhx__med_009': 'Diabetes',
    'ph_p_dhx__med_010': 'Hypertension',
    'ph_p_dhx__med_011': 'Placental Complications',
    'ph_p_dhx__med_012': 'Accident/Injury',
    'ph_p_dhx__alc_001a': 'Alcohol(Pre)',
    'ph_p_dhx__alc_001b': 'Alcohol(During)',
    'ph_p_dhx__coc_001a': 'Cocaine(Pre)',
    'ph_p_dhx__coc_001b': 'Cocaine(During)',
    'ph_p_dhx__mj_001a': 'Marijuana(Pre)',
    'ph_p_dhx__mj_001b': 'Marijuana(During)',
    'ph_p_dhx__nic_001a': 'Tobacco(Pre)',
    'ph_p_dhx__nic_001b': 'Tobacco(During)',
    'ph_p_dhx__opi_001a': 'Opioids(Pre)',
    'ph_p_dhx__opi_001b': 'Opioids(During)',
    'ph_p_dhx__rxpain_001a': 'Rx Opioids(Pre)',
    'ph_p_dhx__rxpain_001b': 'Rx Opioids(During)',
    'ph_p_dhx__vit_001': 'Prenatal Vitamins',
    'screen_time': 'Screen Time',
    'ph_y_mean': 'Physical Activity',
    'sleep_da': 'Arousal Disorder',
    'sleep_dims': 'Sleep Initiation/Maintenance',
    'sleep_does': 'Excessive Sleepiness',
    'sleep_hyphy': 'Sleep Hyperhidrosis',
    'sleep_sbd': 'Sleep Breathing Disorder',
    'sleep_swtd': 'Sleep-Wake Transition',
    'fc_y_srpf__dis_mean': 'School Disengagement',
    'fc_y_srpf__env_mean': 'School Environment',
    'fc_y_srpf__involv_mean': 'School Involvement',
    'fc_p_psb_mean': 'Prosocial Behavior',
    'fc_y_crpbi__cg1_mean': 'Emotional Neglect',
    'fc_y_pm_mean': 'Physical Neglect',
    'nsc_mean': 'Neighborhood Safety',
    'fc_y_fes__confl_mean': 'Family Conflict',
    'ab_g_dyn__cohort_edu__cgs': 'Education(P)',
    'ab_p_demo__income__hhold_001': 'Income(P)',
    'mh_p_kbi__bully_001': 'Bullying',
    'family mentalhealth': 'Family Mentalhealth',
}

MENTAL_MAP = {
    'mh_p_gbi_sum_04A': 'Mania Severity (04A)',
    'mh_y_pps__severity_score_04A': 'Prodromal Psychosis (04A)',
    'mh_y_upps__nurg_sum_04A': 'Negative Urgency (04A)',
    'mh_y_upps__plan_sum_04A': 'Lack of Planning (04A)',
    'mh_y_upps__sens_sum_04A': 'Sensation Seeking (04A)',
    'mh_y_upps__purg_sum_04A': 'Positive Urgency (04A)',
    'mh_y_upps__pers_sum_04A': 'Lack of Perseverance (04A)',
    'mh_p_cbcl__synd__ext_sum_04A': 'Externalizing Problem (04A)',
    'mh_p_cbcl__synd__int_sum_04A': 'Internalizing Problem (04A)',
    'mh_p_cbcl_sum_04A': 'Total Problems (04A)',
    'mh_y_bisbas__bas__dr_sum_04A': 'BAS Drive SUM (04A)',
    'mh_y_bisbas__bas__fs_sum_04A': 'FUN Seeking (04A)',
    'mh_y_bisbas__bas__rr_sum_04A': 'Reward Responsiveness (04A)',
    'mh_y_bisbas__bis_sum_04A': 'BIS Sum (04A)',
    'mh_p_gbi_sum_00A': 'Mania Severity (00A)',
    'mh_y_pps__severity_score_00A': 'Prodromal Psychosis (00A)',
    'mh_p_cbcl__synd__ext_sum_00A': 'Externalizing Problem (00A)',
    'mh_p_cbcl__synd__int_sum_00A': 'Internalizing Problem (00A)',
    'mh_p_cbcl_sum_00A': 'Total Problems (00A)',
}

print(f'LABEL_MAP: {len(LABEL_MAP)} entries')
print(f'MENTAL_MAP: {len(MENTAL_MAP)} entries')

## Step 2 — Define predictor lists

In [ ]:
ENV_PREDICTORS = [
    'Pubertal Stage', 'School Disengagement', 'School Environment', 'School Involvement',
    'Prosocial Behavior', 'Emotional Neglect', 'Physical Neglect', 'Neighborhood Safety',
    'Family Conflict', 'Physical Activity', 'Bullying', 'Screen Time', 'Family Mentalhealth',
    'Arousal Disorder', 'Sleep Initiation/Maintenance', 'Excessive Sleepiness',
    'Sleep Hyperhidrosis', 'Sleep Breathing Disorder', 'Sleep-Wake Transition',
    'Maternal Age', 'Paternal Age', 'Prematurity', 'Cyanosis at Birth', 'Bradycardia at Birth',
    'Apnea at Birth', 'Neonatal Convulsions', 'Jaundice Treatment', 'Oxygen Required',
    'Blood Transfusion', 'Rh Incompatibility', 'Planned Pregnancy', 'Nausea/Vomiting',
    'Heavy Bleeding', 'Pre-eclampsia', 'Gallbladder Attack', 'Proteinuria', 'Rubella',
    'Anemia', 'Infection', 'Diabetes', 'Hypertension', 'Placental Complications',
    'Accident/Injury', 'Alcohol(Pre)', 'Alcohol(During)', 'Cocaine(Pre)', 'Cocaine(During)',
    'Marijuana(Pre)', 'Marijuana(During)', 'Tobacco(Pre)', 'Tobacco(During)',
    'Opioids(Pre)', 'Opioids(During)', 'Rx Opioids(Pre)', 'Rx Opioids(During)',
    'Prenatal Vitamins', 'Breastfeeding Duration', 'Motor Development', 'Speech Development',
    'Income(P)', 'Waist', 'BMI', 'Education(P)',
]

MH_PREDICTORS = [
    'Mania Severity (04A)', 'Prodromal Psychosis (04A)', 'Negative Urgency (04A)',
    'Lack of Planning (04A)', 'Sensation Seeking (04A)', 'Positive Urgency (04A)',
    'Lack of Perseverance (04A)', 'Externalizing Problem (04A)', 'Internalizing Problem (04A)',
    'Total Problems (04A)', 'BAS Drive SUM (04A)', 'FUN Seeking (04A)',
    'Reward Responsiveness (04A)', 'BIS Sum (04A)',
]

ALL_PREDICTORS = ENV_PREDICTORS + MH_PREDICTORS

print(f'ENV_PREDICTORS : {len(ENV_PREDICTORS)}')
print(f'MH_PREDICTORS  : {len(MH_PREDICTORS)}')
print(f'ALL_PREDICTORS : {len(ALL_PREDICTORS)}')

## Step 3 — Load and merge data

In [ ]:
suffix_pat = re.compile(r'\s*\(\d{2}[A-Z]\)$')

env = pd.read_csv(ENV_CSV)
env['ID'] = env['participant_id']
env = env.set_index('ID').rename(columns=LABEL_MAP)

mh  = pd.read_csv(MH_CSV).set_index('ID').rename(columns=MENTAL_MAP)
pca = pd.read_csv(PCA_CSV).set_index('ID')

final = pd.concat([env, mh, pca], axis=1, join='inner')
print(f'Merged dataset shape: {final.shape}')
print(f'Columns present: {list(final.columns[:10])} ...')

## Step 4 — Helper functions

In [ ]:
def to_baseline_cov(name, wave=WAVE_BASE):
    """Strip wave suffix and return the baseline version of a predictor name."""
    base = suffix_pat.sub('', name).strip()
    return f'{base} ({wave})'


def r_to_d(r):
    """Convert Pearson r to Cohen's d."""
    return 2 * r / np.sqrt(max(1e-12, 1 - r**2))


print('Helper functions defined.')

## Step 5 — Partial correlations with PC1_pace

Covariates: `PC1_status`, `Age (Baseline)`, `Sex`, `Ethnicity`, `Race`.
For mental-health predictors that have a baseline counterpart (00A), that baseline score is added as an additional covariate.

In [ ]:
BASE_COVS_PACE = ['PC1_status', 'Age (Baseline)', 'Sex', 'Ethnicity', 'Race']
baseline_mh_names = [v for k, v in MENTAL_MAP.items() if f'({WAVE_BASE})' in v]

rows = []
for pred in ALL_PREDICTORS:
    if pred not in final.columns:
        continue

    covs = BASE_COVS_PACE.copy()
    bc = to_baseline_cov(pred)
    baseline_cov = bc if (bc in baseline_mh_names and bc in final.columns) else None
    if baseline_cov:
        covs.append(baseline_cov)

    active_covs = [c for c in covs if c in final.columns]
    cols = ['PC1_pace', pred] + active_covs
    sub = final[cols].apply(pd.to_numeric, errors='coerce').dropna()
    if len(sub) < 10:
        continue

    pc = pg.partial_corr(
        data=sub, x='PC1_pace', y=pred,
        covar=active_covs, method='pearson'
    )
    r = float(pc['r'].iloc[0])
    p = float(pc['p-val'].iloc[0])
    n = int(pc['n'].iloc[0])
    ci_raw = pc['CI95%'].iloc[0]
    ci = list(ci_raw) if not isinstance(ci_raw, str) else ast.literal_eval(ci_raw)

    rows.append({
        'Predictor': pred,
        'n': n,
        'r': r,
        'CI_low': ci[0],
        'CI_high': ci[1],
        'cohen_d': r_to_d(r),
        'p_val': p,
        'baseline_covariate': baseline_cov,
    })

df_pace = pd.DataFrame(rows)
df_pace['p_fdr'] = multipletests(df_pace['p_val'], method='fdr_bh')[1]
df_pace['sig_fdr'] = df_pace['p_fdr'] < 0.05
df_pace = df_pace.sort_values('p_fdr').reset_index(drop=True)

out_pace = os.path.join(OUTPUT_DIR, 'partial_corr_pace.csv')
df_pace.to_csv(out_pace, index=False)
print(f'PC1_pace results : {df_pace.shape[0]} predictors, '
      f'{df_pace["sig_fdr"].sum()} significant (FDR < 0.05)')
print(f'Saved → {out_pace}')

## Step 6 — Partial correlations with PC1_status

Same procedure as Step 5 but `PC1_pace` is the covariate instead of `PC1_status`, and the target score is `PC1_status`.

In [ ]:
BASE_COVS_STATUS = ['PC1_pace', 'Age (Baseline)', 'Sex', 'Ethnicity', 'Race']

rows_s = []
for pred in ALL_PREDICTORS:
    if pred not in final.columns:
        continue

    covs = BASE_COVS_STATUS.copy()
    bc = to_baseline_cov(pred)
    baseline_cov = bc if (bc in baseline_mh_names and bc in final.columns) else None
    if baseline_cov:
        covs.append(baseline_cov)

    active_covs = [c for c in covs if c in final.columns]
    cols = ['PC1_status', pred] + active_covs
    sub = final[cols].apply(pd.to_numeric, errors='coerce').dropna()
    if len(sub) < 10:
        continue

    pc = pg.partial_corr(
        data=sub, x='PC1_status', y=pred,
        covar=active_covs, method='pearson'
    )
    r = float(pc['r'].iloc[0])
    p = float(pc['p-val'].iloc[0])
    n = int(pc['n'].iloc[0])
    ci_raw = pc['CI95%'].iloc[0]
    ci = list(ci_raw) if not isinstance(ci_raw, str) else ast.literal_eval(ci_raw)

    rows_s.append({
        'Predictor': pred,
        'n': n,
        'r': r,
        'CI_low': ci[0],
        'CI_high': ci[1],
        'cohen_d': r_to_d(r),
        'p_val': p,
        'baseline_covariate': baseline_cov,
    })

df_status = pd.DataFrame(rows_s)
df_status['p_fdr'] = multipletests(df_status['p_val'], method='fdr_bh')[1]
df_status['sig_fdr'] = df_status['p_fdr'] < 0.05
df_status = df_status.sort_values('p_fdr').reset_index(drop=True)

out_status = os.path.join(OUTPUT_DIR, 'partial_corr_status.csv')
df_status.to_csv(out_status, index=False)
print(f'PC1_status results : {df_status.shape[0]} predictors, '
      f'{df_status["sig_fdr"].sum()} significant (FDR < 0.05)')
print(f'Saved → {out_status}')

## Step 7 — Summary: top predictors

In [ ]:
TOP_N = 10

print('=' * 60)
print(f'Top {TOP_N} significant predictors of PC1_pace (FDR < 0.05)')
print('=' * 60)
sig_pace = df_pace[df_pace['sig_fdr']].head(TOP_N)
if len(sig_pace):
    print(sig_pace[['Predictor', 'n', 'r', 'cohen_d', 'p_val', 'p_fdr']].to_string(index=False))
else:
    print('No significant predictors after FDR correction.')

print()
print('=' * 60)
print(f'Top {TOP_N} significant predictors of PC1_status (FDR < 0.05)')
print('=' * 60)
sig_status = df_status[df_status['sig_fdr']].head(TOP_N)
if len(sig_status):
    print(sig_status[['Predictor', 'n', 'r', 'cohen_d', 'p_val', 'p_fdr']].to_string(index=False))
else:
    print('No significant predictors after FDR correction.')